# 🎬 Урок 1. OpenCV: картинка → кадр → видео

**Что будет сегодня**
1. **Часть 1.** Учимся работать с картинкой в OpenCV (`cv2`): открыть, показать, узнать размер, обрезать, повернуть, нарисовать. Видео — это много картинок, поэтому всё это пригодится.
2. **Часть 2.** Открываем видео, достаём из него кадры, смотрим прямо в Colab, меняем каждый кадр и сохраняем.

**Как работать**
* Сначала: **Файл → Сохранить копию на Диске**, иначе ваши изменения пропадут.
* Запуск ячейки: **Shift + Enter**. Запускайте ячейки по порядку сверху вниз.
* В заданиях ✏️ замените `None` на свой код. Под каждым заданием есть проверка ✅ / ❌ и две подсказки: 💡 — идея, 🆘 — почти готовый ответ.
* Шпаргалка по всем функциям — в самом конце ноутбука.

In [ ]:
#@title ⚙️ Запустите эту ячейку первой (подготовка) { display-mode: "form" }
# Здесь подключаются библиотеки и несколько функций-помощников:
#   download(имя)        — скачать учебный файл (картинку или видео)
#   cv2_imshow(картинка) — показать картинку в Colab
#   show_video(кадры)    — показать список кадров как видео
#   upload_video()       — загрузить своё видео с компьютера
#   check(...)           — проверка задания ✅ / ❌
import base64, os, shutil, subprocess, tempfile, urllib.request
import cv2
import numpy as np
from IPython.display import HTML, display

try:
    from google.colab import files
    from google.colab.patches import cv2_imshow as _colab_imshow
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    _colab_imshow = None

def cv2_imshow(img):
    """Показывает картинку. Если вместо картинки None — подсказывает, а не падает."""
    if img is None:
        print("⚠️ Показывать нечего: переменная пока None. Выполните задание (или проверьте имя файла)")
        return
    if _colab_imshow:
        _colab_imshow(img)
        return
    img = np.clip(np.asarray(img), 0, 255).astype(np.uint8)
    b64 = base64.b64encode(cv2.imencode(".png", img)[1]).decode()
    display(HTML(f'<img src="data:image/png;base64,{b64}">'))

def download(name):
    """Скачивает учебный файл из примеров OpenCV. Качает сервер Google, а не ваш интернет."""
    if not os.path.exists(name):
        urllib.request.urlretrieve("https://raw.githubusercontent.com/opencv/opencv/4.x/samples/data/" + name, name)
    print(f"Файл {name} готов ({os.path.getsize(name) // 1024} КБ)")

def _mp4(frames, fps):
    if not shutil.which("ffmpeg"):
        return None
    h, w = frames[0].shape[:2]
    cmd = ["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "bgr24", "-s", f"{w}x{h}", "-r", str(fps),
           "-i", "-", "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "30", "-preset", "veryfast",
           "-movflags", "+faststart", "-f", "mp4", "-"]
    res = subprocess.run(cmd, input=b"".join(f.tobytes() for f in frames), capture_output=True)
    return res.stdout if res.returncode == 0 and res.stdout else None

def show_video(frames, fps=10, width=480):
    """В Colab нет окна для видео. Поэтому склеиваем кадры в маленький mp4 и показываем плеером."""
    if len(frames) == 0 or any(f is None for f in frames):
        print("⚠️ Показывать нечего: список пуст или в нём None вместо кадров. Выполните задание")
        return
    prep = []
    for f in frames:
        f = np.clip(np.asarray(f), 0, 255).astype(np.uint8)
        if f.ndim == 2:                       # серый кадр → три одинаковых канала
            f = cv2.cvtColor(f, cv2.COLOR_GRAY2BGR)
        h, w = f.shape[:2]
        prep.append(np.ascontiguousarray(f[: h - h % 2, : w - w % 2]))
    data = _mp4(prep, fps)
    if data:
        b64 = base64.b64encode(data).decode()
        display(HTML(f'<video width="{width}" controls autoplay loop muted playsinline>'
                     f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))
    else:
        print("ffmpeg не найден — показываю первый и последний кадр")
        cv2_imshow(np.hstack([prep[0], prep[-1]]))

def upload_video(max_frames=80, width=384):
    """Загрузка своего видео. Берём не больше max_frames кадров и уменьшаем их до ширины width."""
    if not IN_COLAB:
        print("Загрузка работает только в Colab")
        return [], 10
    name = next(iter(files.upload()))
    cap = cv2.VideoCapture(name)
    fps = cap.get(cv2.CAP_PROP_FPS) or 10
    frames = []
    while len(frames) < max_frames:
        ok, frame = cap.read()
        if not ok:
            break
        h, w = frame.shape[:2]
        frames.append(cv2.resize(frame, (width, int(h * width / w))))
    cap.release()
    print(f"Прочитано кадров: {len(frames)}, FPS: {fps:.0f}")
    return frames, fps

def _ok(msg):
    display(HTML(f'<div style="font:700 16px sans-serif;color:#1d8a52;margin:6px 0">✅ {msg}</div>'))

def _fail(msg):
    display(HTML(f'<div style="font:700 16px sans-serif;color:#d93a3f;margin:6px 0">❌ {msg} — откройте подсказку под заданием</div>'))

def check(cond, good, bad="Пока не получилось"):
    """check(условие, "текст если верно", "текст если нет")"""
    try:
        result = bool(cond())
    except Exception:
        result = False
    _ok(good) if result else _fail(bad)
    return result

print("✅ Готово! OpenCV версии", cv2.__version__)

---
# Часть 1. Картинка в OpenCV

## 1. Открываем и показываем картинку
* `cv2.imread("имя файла")` — читает картинку из файла;
* `cv2_imshow(картинка)` — показывает её в Colab.

Сначала скачаем две учебные картинки.

In [ ]:
download("fruits.jpg")
download("baboon.jpg")

img = cv2.imread("fruits.jpg")   # читаем картинку в переменную img
cv2_imshow(img)                  # показываем

### ✏️ Задание 1. Откройте и покажите вторую картинку — `baboon.jpg`

In [ ]:
img2 = None   # ← замените None
# ↓ и покажите img2

check(lambda: img2.shape == (512, 512, 3), "Картинка открыта!")

<details><summary>💡 Подсказка</summary>

Делаем то же, что в примере выше, только с другим именем файла. Имя пишется в кавычках.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
img2 = cv2.imread("___")
cv2_imshow(___)
```
</details>

> ⚠️ Если в имени файла ошибка, `cv2.imread` **не выдаёт ошибку**, а возвращает `None`. Ошибка вылезет позже, например «'NoneType' object has no attribute 'shape'». Увидели такое — проверьте имя файла.

## 2. Размер картинки — `shape`
Картинка в OpenCV — это **таблица чисел** (массив numpy). У неё есть форма `img.shape`:

**(высота, ширина, каналы)**

Каналов 3: синий, зелёный, красный.

In [ ]:
print(img.shape)

### ✏️ Задание 2. Достаньте высоту, ширину и число каналов по отдельности
`img.shape` — это кортеж из трёх чисел. Элемент кортежа достают по номеру в квадратных скобках, нумерация с **0**.

In [ ]:
h = None   # высота
w = None   # ширина
c = None   # каналы

print("высота:", h, "ширина:", w, "каналы:", c)
check(lambda: (h, w, c) == img.shape, "Верно! Запомните порядок: сначала высота, потом ширина")

<details><summary>💡 Подсказка</summary>

Высота — первый элемент `img.shape`, то есть номер 0. Ширина — номер 1, каналы — номер 2.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
h = img.shape[0]
w = img.shape[___]
c = img.shape[___]
```
</details>

### ✏️ Задание 3. Сколько всего пикселей в картинке?

In [ ]:
pixels = None   # ← используйте h и w

print("Пикселей:", pixels)
check(lambda: pixels == img.shape[0] * img.shape[1], f"Да! {pixels} пикселей, и в каждом по 3 числа")

<details><summary>💡 Подсказка</summary>

Картинка — прямоугольник из пикселей. Сколько клеток в прямоугольнике высотой h и шириной w?
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
pixels = h * ___
```
</details>

## 3. Из каких чисел состоит картинка
* `img.dtype` — тип чисел. `uint8` — целые от **0 до 255**.
* `img.min()` и `img.max()` — самое маленькое и самое большое число в картинке.

In [ ]:
print("тип:", img.dtype)
print("минимум:", img.min(), " максимум:", img.max())

### ✏️ Задание 4. Найдите самое большое число в картинке `img2`

In [ ]:
biggest = None

print("Максимум в img2:", biggest)
check(lambda: biggest == img2.max(), "Верно!")

<details><summary>💡 Подсказка</summary>

Делаем как в примере выше, только для `img2`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
biggest = img2.___()
```
</details>

## 4. Один пиксель
К пикселю обращаются так: `img[y, x]` — сначала **строка** (y, сверху вниз), потом **столбец** (x, слева направо).

В ответ приходят три числа: **[синий, зелёный, красный]**. OpenCV хранит цвета в порядке **BGR**, а не RGB — запомните это.

In [ ]:
print("Пиксель в строке 10, столбце 20:", img[10, 20])

### ✏️ Задание 5. Какого цвета пиксель в самом центре картинки?
Центр — это половина высоты и половина ширины. Делим нацело: `//`.

In [ ]:
center = None

print("Пиксель в центре (B, G, R):", center)
check(lambda: (center == img[img.shape[0] // 2, img.shape[1] // 2]).all(), "Нашли центр!")

<details><summary>💡 Подсказка</summary>

Строка центра — `h // 2`, столбец — `w // 2`. Сначала строка, потом столбец.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
center = img[h // 2, ___]
```
</details>

## 5. Перекрашиваем кусок картинки
Можно записать цвет сразу в целый прямоугольник: `картинка[строки, столбцы] = (B, G, R)`.
* `0:50` — строки (или столбцы) с 0 по 49;
* `:` — все строки (или все столбцы).

Перед изменением делаем копию `.copy()`, чтобы не испортить оригинал.

In [ ]:
pic = img.copy()
pic[0:50, 0:50] = (0, 0, 0)        # чёрный квадрат 50×50 в левом верхнем углу
cv2_imshow(pic)

### ✏️ Задание 6. Нарисуйте красную полосу высотой 20 пикселей через весь верх картинки
Красный в BGR — это `(0, 0, 255)`.

In [ ]:
stripe = img.copy()
# ↓ ваш код: строки 0:20, все столбцы, красный цвет

cv2_imshow(stripe)
check(lambda: (stripe[0:20, :] == (0, 0, 255)).all() and (stripe[20:] == img[20:]).all(), "Красная полоса на месте!")

<details><summary>💡 Подсказка</summary>

Строки — `0:20`, столбцы — все, то есть `:`. Цвет — `(0, 0, 255)`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
stripe[0:20, ___] = (0, 0, 255)
```
</details>

## 6. Обрезка (вырезаем кусок)
Тот же срез, только без `=`: `кусок = img[строки, столбцы]`.

In [ ]:
piece = img[100:300, 150:350]       # строки 100–299, столбцы 150–349
print("Размер куска:", piece.shape)
cv2_imshow(piece)

### ✏️ Задание 7. Вырежьте левую половину картинки
Все строки, а столбцы — от 0 до половины ширины.

In [ ]:
left = None

cv2_imshow(left)
check(lambda: left.shape == (img.shape[0], img.shape[1] // 2, 3), "Левая половина вырезана!")

<details><summary>💡 Подсказка</summary>

Строки — все (`:`). Столбцы — от 0 до `w // 2`, пишется `0:w // 2` или короче `:w // 2`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
left = img[:, 0:___]
```
</details>

## 7. Изменение размера — `cv2.resize`
`cv2.resize(картинка, (ширина, высота))`

⚠️ **Ловушка:** в `resize` сначала **ширина**, потом высота. А в `shape` — наоборот, сначала высота.

In [ ]:
small = cv2.resize(img, (200, 100))   # ширина 200, высота 100
print("Было:", img.shape, " стало:", small.shape)
cv2_imshow(small)

### ✏️ Задание 8. Уменьшите картинку ровно в 2 раза

In [ ]:
half = None

cv2_imshow(half)
check(lambda: half.shape == (img.shape[0] // 2, img.shape[1] // 2, 3), "Ровно в 2 раза меньше!")

<details><summary>💡 Подсказка</summary>

Новая ширина — `w // 2`, новая высота — `h // 2`. В `resize` сначала ширина!
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
half = cv2.resize(img, (___, ___))
```
</details>

## 8. Отражение — `cv2.flip`
`cv2.flip(картинка, код)`:
* `1` — отражение влево-вправо (как в зеркале);
* `0` — отражение вверх-вниз;
* `-1` — и то и другое сразу.

In [ ]:
mirror = cv2.flip(img, 1)
cv2_imshow(np.hstack([img, mirror]))   # np.hstack ставит картинки рядом

### ✏️ Задание 9. Переверните картинку на 180° (вверх ногами)
Попробуйте коды `0` и `-1`. Какой из них даёт поворот на 180°? Это понадобится на контрольной.

In [ ]:
upside = None

cv2_imshow(upside)
check(lambda: (upside == np.rot90(img, 2)).all(), "Это поворот на 180°!",
      "Это ещё не поворот на 180°: код 0 только переворачивает, но картинка остаётся зеркальной")

<details><summary>💡 Подсказка</summary>

Поворот на 180° = отражение вверх-вниз **и** влево-вправо сразу.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
upside = cv2.flip(img, ___)
```
</details>

## 9. Поворот на 90° — `cv2.rotate`
* `cv2.ROTATE_90_CLOCKWISE` — по часовой стрелке;
* `cv2.ROTATE_90_COUNTERCLOCKWISE` — против часовой;
* `cv2.ROTATE_180` — на 180°.

In [ ]:
turned = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
print("Было:", img.shape, " стало:", turned.shape)   # высота и ширина поменялись местами!
cv2_imshow(turned)

### ✏️ Задание 10. Поверните картинку против часовой стрелки

In [ ]:
ccw = None

cv2_imshow(ccw)
check(lambda: (ccw == np.rot90(img, 1)).all(), "Повернули против часовой!")

<details><summary>💡 Подсказка</summary>

Нужна та же функция, но с другой константой из списка выше.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
ccw = cv2.rotate(img, cv2.ROTATE_90____)
```
</details>

## 10. Чёрно-белая картинка — `cv2.cvtColor`
`cv2.cvtColor(картинка, способ)` переводит из одной цветовой модели в другую. `cv2.COLOR_BGR2GRAY` означает «из BGR в серый».

У серой картинки **нет каналов**: на пиксель приходится одно число, яркость.

In [ ]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
print("Цветная:", img.shape, " серая:", gray.shape)
cv2_imshow(gray)

### ✏️ Задание 11. Сделайте серой картинку `img2` и узнайте яркость её центрального пикселя

In [ ]:
gray2 = None
bright = None    # img2 размером 512×512, центр — [256, 256]

print("Яркость центра:", bright)
check(lambda: gray2.ndim == 2 and bright == gray2[256, 256], "Готово! Одно число вместо трёх")

<details><summary>💡 Подсказка</summary>

Первая строка — как в примере, только с `img2`. Яркость — это просто `gray2[строка, столбец]`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
gray2 = cv2.cvtColor(___, cv2.COLOR_BGR2GRAY)
bright = gray2[256, ___]
```
</details>

## 11. Каналы B, G, R по отдельности
`b, g, r = cv2.split(img)` раскладывает картинку на три серые: каждая показывает, сколько в пикселе синего, зелёного, красного. Чем светлее, тем больше этого цвета.

`cv2.merge([b, g, r])` собирает каналы обратно.

In [ ]:
b, g, r = cv2.split(img)
cv2_imshow(np.hstack([b, g, r]))   # слева синий, в центре зелёный, справа красный

### ✏️ Задание 12. Соберите картинку, поменяв местами синий и красный
Что стало с цветами? Так выглядит картинка, если перепутать BGR и RGB.

In [ ]:
swapped = None

cv2_imshow(swapped)
check(lambda: (swapped == img[:, :, ::-1]).all(), "Апельсины посинели! Вот почему важен порядок BGR")

<details><summary>💡 Подсказка</summary>

В `cv2.merge` передаётся список каналов. Обычный порядок `[b, g, r]` — поменяйте первый и последний.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
swapped = cv2.merge([___, g, ___])
```
</details>

## 12. Яркость и ловушка переполнения
Хочется сделать ярче — просто прибавить 100? Посмотрите, что будет: `uint8` не умеет больше 255, и число «перескакивает» через ноль (250 + 10 = 4).

Правильно делать через `cv2.convertScaleAbs(картинка, alpha=контраст, beta=яркость)`: результат обрезается до 0…255.

In [ ]:
wrong = img + 100                                          # ❌ светлые места стали тёмными
right = cv2.convertScaleAbs(img, alpha=1.0, beta=100)      # ✅ ярче без ошибок
cv2_imshow(np.hstack([img, wrong, right]))

### ✏️ Задание 13. Сделайте картинку темнее на 60

In [ ]:
dark = None

cv2_imshow(dark)
check(lambda: (dark == cv2.convertScaleAbs(img, alpha=1.0, beta=-60)).all(), "Стало темнее!")

<details><summary>💡 Подсказка</summary>

Темнее — значит `beta` с минусом.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
dark = cv2.convertScaleAbs(img, alpha=1.0, beta=___)
```
</details>

### ✏️ Задание 14. Усильте **только красный** канал на 10% (это задание контрольной!)
Красный канал — `img[:, :, 2]`. Умножить на 1.1 = `alpha=1.1`.

In [ ]:
warm = img.copy()
# ↓ ваш код: warm[:, :, 2] = ...

cv2_imshow(np.hstack([img, warm]))
check(lambda: (warm[:, :, 2] == cv2.convertScaleAbs(img[:, :, 2], alpha=1.1)).all() and (warm[:, :, :2] == img[:, :, :2]).all(),
      "Картинка стала теплее! На контрольной это будет по кнопке s")

<details><summary>💡 Подсказка</summary>

Берём красный канал `img[:, :, 2]`, пропускаем через `convertScaleAbs` с `alpha=1.1` и записываем обратно в `warm[:, :, 2]`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
warm[:, :, 2] = cv2.convertScaleAbs(img[:, :, ___], alpha=___)
```
</details>

## 13. Рисуем на картинке
| Что | Как |
|---|---|
| линия | `cv2.line(img, (x1, y1), (x2, y2), цвет, толщина)` |
| прямоугольник | `cv2.rectangle(img, (x1, y1), (x2, y2), цвет, толщина)` |
| круг | `cv2.circle(img, (x, y), радиус, цвет, толщина)` |
| текст | `cv2.putText(img, "текст", (x, y), шрифт, размер, цвет, толщина)` |

Координаты здесь — **(x, y)**: сначала по горизонтали. Толщина `-1` означает «залить». ⚠️ `cv2.putText` не умеет русские буквы — пишем латиницей.

In [ ]:
art = img.copy()
cv2.line(art, (0, 0), (200, 100), (255, 0, 0), 3)                  # синяя линия
cv2.rectangle(art, (250, 50), (450, 200), (0, 255, 0), 4)          # зелёная рамка
cv2.circle(art, (100, 350), 50, (0, 0, 255), -1)                   # красный круг (залит)
cv2.putText(art, "Hello, OpenCV!", (150, 450), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3)
cv2_imshow(art)

### ✏️ Задание 15. Оформите картинку
На копии `img`:
1. рамка по всему краю картинки, толщина 10, любой цвет;
2. залитый круг радиусом 30 в центре;
3. ваше имя латиницей внизу.

In [ ]:
poster = img.copy()
# ↓ ваш код: рамка, круг, имя


cv2_imshow(poster)
check(lambda: (poster[2, w // 2] != img[2, w // 2]).any() and (poster[h - 3, w // 3] != img[h - 3, w // 3]).any()
      and (poster[h // 2, w // 2] != img[h // 2, w // 2]).any(), "Красиво! Рамка и круг на месте")

<details><summary>💡 Подсказка</summary>

Рамка — прямоугольник от `(0, 0)` до `(w - 1, h - 1)`. Центр — `(w // 2, h // 2)`. Залить — толщина `-1`. Текст — точка `(20, h - 25)`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
cv2.rectangle(poster, (0, 0), (w - 1, h - 1), (60, 200, 255), ___)
cv2.circle(poster, (w // 2, h // 2), 30, (255, 0, 255), ___)
cv2.putText(poster, "Name", (20, h - 25), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3)
```
</details>

## 14. Сохраняем картинку — `cv2.imwrite`
`cv2.imwrite("имя.jpg", картинка)`. Файл появится слева во вкладке 📁 «Файлы».

### ✏️ Задание 16. Сохраните свой постер в файл `poster.jpg`

In [ ]:
# ↓ ваш код

check(lambda: os.path.exists("poster.jpg"), "Файл poster.jpg сохранён! Найдите его в 📁 слева")

<details><summary>💡 Подсказка</summary>

Первым идёт имя файла в кавычках, вторым — картинка.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
cv2.imwrite("poster.jpg", ___)
```
</details>

---
# Часть 2. Видео в OpenCV
**Видео — это много картинок (кадров) подряд.** Всё, что вы сделали с картинкой, можно сделать с каждым кадром.

## 15. Открываем видео — `cv2.VideoCapture`
Скачаем учебное видео с пешеходами и откроем его. `VideoCapture` ничего не загружает в память сразу: он как «проигрыватель», из которого мы будем доставать кадры по одному.

In [ ]:
download("vtest.avi")

cap = cv2.VideoCapture("vtest.avi")
print("Видео открылось?", cap.isOpened())

## 16. Свойства видео — `cap.get`

In [ ]:
fps = cap.get(cv2.CAP_PROP_FPS)                 # кадров в секунду
count = cap.get(cv2.CAP_PROP_FRAME_COUNT)       # всего кадров
width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)       # ширина кадра
height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)     # высота кадра
print("FPS:", fps, " кадров:", count, " размер:", width, "×", height)

### ✏️ Задание 17. Сколько секунд длится видео?

In [ ]:
duration = None

print("Длительность:", duration, "с")
check(lambda: abs(duration - count / fps) < 1e-6, f"Верно! {count:.0f} кадров / {fps:.0f} FPS")

<details><summary>💡 Подсказка</summary>

FPS — сколько кадров в одной секунде. Если всего `count` кадров, сколько это секунд?
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
duration = count / ___
```
</details>

## 17. Достаём кадр — `cap.read()`
`cap.read()` возвращает **два** значения:
* `ok` — получилось ли прочитать (`True` / `False`, в конце видео будет `False`);
* `frame` — сам кадр, обычная картинка, как в части 1.

Каждый следующий вызов `read()` даёт **следующий** кадр.

In [ ]:
ok, frame = cap.read()
print("Прочитали?", ok, " размер кадра:", frame.shape)
cv2_imshow(frame)

### ✏️ Задание 18. Прочитайте следующий кадр и покажите его

In [ ]:
ok2, frame2 = None, None   # ← прочитайте кадр так же, как выше

cv2_imshow(frame2)
check(lambda: ok2 and frame2.shape == frame.shape, "Это уже второй кадр! Люди сдвинулись совсем чуть-чуть")

<details><summary>💡 Подсказка</summary>

Скопируйте строку `ok, frame = cap.read()` из примера и поменяйте имена переменных.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
ok2, frame2 = cap.___()
```
</details>

## 18. Кадр по номеру — `cap.set`
Можно перемотать на нужный кадр: `cap.set(cv2.CAP_PROP_POS_FRAMES, номер)`, а потом сделать `read()`.

In [ ]:
cap.set(cv2.CAP_PROP_POS_FRAMES, 300)   # перематываем на кадр № 300
ok, frame300 = cap.read()
cv2_imshow(frame300)

### ✏️ Задание 19. Покажите кадр из середины видео
Номер кадра должен быть целым числом: `int(...)`.

In [ ]:
middle = 0      # ← замените 0 на номер среднего кадра
cap.set(cv2.CAP_PROP_POS_FRAMES, middle)
ok, frame_mid = cap.read()

cv2_imshow(frame_mid)
check(lambda: middle == int(count // 2), f"Кадр № {middle} — ровно середина")

<details><summary>💡 Подсказка</summary>

Середина — половина от общего числа кадров `count`. Делим нацело и превращаем в `int`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
middle = int(count // ___)
```
</details>

## 19. Читаем много кадров в цикле
Вот главный цикл работы с видео. Разберём построчно:

In [ ]:
cap = cv2.VideoCapture("vtest.avi")   # открываем видео заново (с начала)
frames = []                           # сюда сложим кадры

while len(frames) < 80:               # берём первые 80 кадров (8 секунд)
    ok, frame = cap.read()            # читаем очередной кадр
    if not ok:                        # видео кончилось — выходим
        break
    frame = cv2.resize(frame, (384, 288))   # уменьшаем: быстрее считать и показывать
    frames.append(frame)              # кладём кадр в список

cap.release()                         # закрываем видео
print("Прочитано кадров:", len(frames))

## 20. Смотрим видео прямо в Colab
У Colab нет окна для видео, поэтому функция-помощник `show_video` склеивает кадры в маленький ролик и показывает его плеером.

In [ ]:
show_video(frames, fps=10)

### ✏️ Задание 20. Сколько секунд длится наш кусочек из 80 кадров?

In [ ]:
clip_seconds = None

check(lambda: abs(clip_seconds - len(frames) / fps) < 1e-6, f"Да, {len(frames) / fps:.0f} секунд")

<details><summary>💡 Подсказка</summary>

Как в задании 17: число кадров `len(frames)` делим на FPS.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
clip_seconds = len(frames) / ___
```
</details>

## 21. Меняем каждый кадр
Чтобы обработать видео, берём кадры по одному, меняем и складываем в новый список. Пример — зеркальное видео:

In [ ]:
result = []
for frame in frames:             # для каждого кадра...
    new = cv2.flip(frame, 1)     # ...отражаем его
    result.append(new)           # и кладём в результат

show_video(result)

### ✏️ Задание 21. Сделайте видео чёрно-белым
Внутри цикла переведите кадр в серый (как в задании 11).

In [ ]:
gray_video = []
for frame in frames:
    new = None        # ← серый кадр
    gray_video.append(new)

show_video(gray_video)
check(lambda: len(gray_video) == len(frames) and gray_video[0].ndim == 2, "Чёрно-белое кино готово!")

<details><summary>💡 Подсказка</summary>

Внутри цикла та же строка, что в задании 11, только для `frame`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
    new = cv2.cvtColor(frame, cv2.___)
```
</details>

### ✏️ Задание 22. Напишите номер кадра на каждом кадре
`enumerate(frames)` даёт сразу номер и кадр: `for i, frame in enumerate(frames):`. Не забудьте `.copy()`, чтобы не испортить исходные кадры.

In [ ]:
numbered = []
for i, frame in enumerate(frames):
    new = frame.copy()
    # ↓ ваш код: cv2.putText(new, ...) с текстом f"frame {i}"

    numbered.append(new)

show_video(numbered)
check(lambda: (numbered[5] != frames[5]).any() and len(numbered) == len(frames), "Номера появились!")

<details><summary>💡 Подсказка</summary>

`f"frame {i}"` — строка, в которую подставляется номер. Позиция — например `(10, 30)`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
    cv2.putText(new, f"frame {i}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), ___)
```
</details>

### ✏️ Задание 23. Переверните всё видео на 180°

In [ ]:
upside_video = []   # ← напишите цикл, как в примерах выше

show_video(upside_video)
check(lambda: len(upside_video) == len(frames) and (upside_video[0] == cv2.flip(frames[0], -1)).all(), "Мир вверх ногами!")

<details><summary>💡 Подсказка</summary>

Скопируйте цикл из примера с зеркалом. Код поворота на 180° — из задания 9.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
upside_video = []
for frame in frames:
    upside_video.append(cv2.flip(frame, ___))
```
</details>

## 22. Сохраняем видео в файл — `cv2.VideoWriter`
`VideoWriter` — «записыватель». Ему нужно знать имя файла, формат сжатия (кодек), FPS и размер кадра `(ширина, высота)`. Дальше кладём в него кадры через `write`.

In [ ]:
h, w = result[0].shape[:2]
out = cv2.VideoWriter("mirror.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
for frame in result:
    out.write(frame)
out.release()                       # обязательно закрыть, иначе файл будет битым

print("Сохранено:", os.path.getsize("mirror.mp4") // 1024, "КБ")
if IN_COLAB:
    files.download("mirror.mp4")    # скачать к себе на компьютер

### ✏️ Задание 24. Сохраните видео с номерами кадров в файл `numbered.mp4`
Скопируйте пример выше и поменяйте имя файла и список кадров.

In [ ]:
# ↓ ваш код

check(lambda: os.path.getsize("numbered.mp4") > 10000, "Видео сохранено!")

<details><summary>💡 Подсказка</summary>

Нужно поменять две вещи: имя `"mirror.mp4"` → `"numbered.mp4"` и список `result` → `numbered`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
h, w = numbered[0].shape[:2]
out = cv2.VideoWriter("numbered.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
for frame in ___:
    out.write(frame)
out.release()
```
</details>

## 23. По желанию: своё видео
Можно загрузить свой **короткий** ролик (до 5 секунд, лучше снятый не в 4K): интернет у нас общий на всех. После загрузки повторите с ним задания 21–23: просто замените `frames` на `my_frames`.

In [ ]:
# my_frames, my_fps = upload_video()
# show_video(my_frames, fps=my_fps)

---
# Как это выглядит на обычном компьютере (не в Colab)
На компьютере с установленным OpenCV видео можно показывать в окне и брать прямо с веб-камеры. Запускать здесь не нужно — в Colab нет окна. Но **именно по этому шаблону будет контрольная**: клавиша включает и выключает эффект.

```python
import cv2

cap = cv2.VideoCapture(0)              # 0 — веб-камера
flipped = False                        # включён ли эффект

while True:
    ok, frame = cap.read()             # 1. читаем кадр
    if not ok:
        break

    key = cv2.waitKey(1) & 0xFF        # какая клавиша нажата (ждём 1 мс)
    if key == ord("t"):                # нажали t — переключаем эффект
        flipped = not flipped
    if key == ord("q"):                # q — выход
        break

    if flipped:                        # 2. обрабатываем кадр
        frame = cv2.flip(frame, -1)

    cv2.imshow("video", frame)         # 3. показываем в окне

cap.release()
cv2.destroyAllWindows()
```
`not flipped` переворачивает значение: `False → True → False`. Так одна и та же клавиша включает и выключает эффект.

---
# 🏠 Домашнее задание (КТ 1): «Свой видео-продакшн»
Делаете в этом же ноутбуке ниже, на кадрах `frames`.
**Сдать в LXP:** ссылку на ноутбук (Настройки доступа → «Все, у кого есть ссылка») и файл `production.mp4`.

| № | Что сделать | Баллы |
|---|---|---|
| 1 | На **каждом** кадре — ваша подпись (латиницей) и **логотип**: цветной квадрат 50×50 с буквой, вставленный в правый верхний угол срезом (как в задании 6). Логотип появляется **только со 2-й секунды** | 1 |
| 2 | **Таймер с тенью**: в левом верхнем углу время от начала видео `00:03`. Тень — тот же текст тёмным цветом со сдвигом на 2 пикселя, рисуется **первой**. Шрифт и цвет — не такие, как в примерах | 1 |

<details><summary>💡 Подсказки</summary>

* Номер кадра `i` берём из `enumerate`. Секунда кадра: `sec = int(i / fps)`.
* Логотип со 2-й секунды: `if i >= 2 * fps:`.
* Вставить логотип: `new[10:60, w - 60:w - 10] = logo` — размер куска должен совпадать с размером логотипа 50×50.
* Время строкой: `f"{sec // 60:02d}:{sec % 60:02d}"` → `00:03`.
* Тень: сначала `putText` тёмным цветом в точке `(12, 32)`, потом светлым в `(10, 30)`.
* Шрифты: `cv2.FONT_HERSHEY_DUPLEX`, `cv2.FONT_HERSHEY_COMPLEX`, `cv2.FONT_HERSHEY_TRIPLEX`.
</details>

In [ ]:
# Заготовка логотипа: жёлтый квадрат 50×50 с буквой
logo = np.full((50, 50, 3), (60, 200, 255), np.uint8)
cv2.putText(logo, "A", (12, 38), cv2.FONT_HERSHEY_DUPLEX, 1.2, (40, 30, 30), 2)
cv2_imshow(logo)

production = []
for i, frame in enumerate(frames):
    new = frame.copy()
    h, w = new.shape[:2]
    # 1. подпись и логотип (со 2-й секунды)

    # 2. таймер с тенью

    production.append(new)

show_video(production)

In [ ]:
# Сохранение результата
h, w = production[0].shape[:2]
out = cv2.VideoWriter("production.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
for frame in production:
    out.write(frame)
out.release()
if IN_COLAB:
    files.download("production.mp4")

---
# 📋 Шпаргалка cv2

| Задача | Код |
|---|---|
| открыть картинку | `img = cv2.imread("file.jpg")` |
| показать в Colab | `cv2_imshow(img)` |
| размер | `h, w, c = img.shape` (высота, ширина, каналы) |
| пиксель | `img[y, x]` → `[B, G, R]` |
| закрасить кусок | `img[y1:y2, x1:x2] = (B, G, R)` |
| вырезать кусок | `piece = img[y1:y2, x1:x2]` |
| копия | `img.copy()` |
| изменить размер | `cv2.resize(img, (ширина, высота))` |
| отразить | `cv2.flip(img, 1)` ↔, `0` ↕, `-1` = 180° |
| повернуть на 90° | `cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)` |
| серый | `cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)` |
| каналы | `b, g, r = cv2.split(img)` · `cv2.merge([b, g, r])` |
| яркость | `cv2.convertScaleAbs(img, alpha=1.0, beta=50)` |
| линия / рамка / круг | `cv2.line` · `cv2.rectangle` · `cv2.circle` |
| текст | `cv2.putText(img, "text", (x, y), cv2.FONT_HERSHEY_SIMPLEX, 1, (B, G, R), 2)` |
| сохранить картинку | `cv2.imwrite("out.jpg", img)` |
| открыть видео | `cap = cv2.VideoCapture("video.mp4")` (или `0` — камера) |
| свойства видео | `cap.get(cv2.CAP_PROP_FPS)` · `CAP_PROP_FRAME_COUNT` · `CAP_PROP_FRAME_WIDTH` |
| прочитать кадр | `ok, frame = cap.read()` |
| перемотать | `cap.set(cv2.CAP_PROP_POS_FRAMES, n)` |
| записать видео | `cv2.VideoWriter("out.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))` → `out.write(frame)` → `out.release()` |
| показать видео в Colab | `show_video(frames)` |